# Qwen3-0.6B × Triton split-KV (FlashDecoding) AttentionBackend

**vllm_split_v2** — vllm_split 위에 **decode 경로의 split-KV** 만 추가한 변형. prefill 은 동일.

Decode 가 두 단계로 분리됨:
- **phase-1 (`_fwd_kernel_decode_partial`)**: KV 를 KV_SPLITS 개 chunk 로 나눠 각 program 이 자기 chunk 에 대한 partial m / l / acc 를 계산
- **phase-2 (`_fwd_kernel_decode_reduce`)**: log-sum-exp 안정 결합으로 partial 들을 하나의 final O 로 통합

Tri Dao et al. *FlashDecoding* (2023.10) 의 schedule layer 가 여기에 해당. 알고리즘 (online softmax) 은 그대로, **grid 에 KV_SPLITS 차원만 추가**.

**범위 고지**
- KV_SPLITS 는 wrapper 가 자동 결정 (`grid_bh < 2 × num_SMs` 일 때 split, 그 외엔 simple kernel fast path)
- `MY_TRITON_BACKEND_LOG=1` 시 첫 호출에 `MyTritonImpl[v2].forward fired (decode, split-KV)` 로그 1회
- `max_num_seqs=1`, `enforce_eager=True` 단일 시퀀스 한정

## 준비물 & 설치

- CUDA GPU 필수
- Python >= 3.10
- vLLM **0.19.1 정확히** 권장 (내부 API drift)

```bash
# 노트북 디렉토리에서
pip install -e .
```

`pyproject.toml` 의 entry point (`vllm.general_plugins = triton_attention_backend:register`) 가
모든 vLLM 프로세스에서 자동으로 `register()` 를 호출한다.

**같은 venv 에 vllm_split (또는 다른 stage) 와 동시 설치 금지** — `py-modules` 이름이 같아 충돌.

In [ ]:
import torch, triton, vllm

print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('vllm:', vllm.__version__)
print('triton:', triton.__version__)

if not torch.cuda.is_available():
    raise SystemExit('이 노트북은 CUDA GPU가 필요합니다.')

assert vllm.__version__.startswith('0.19'), (
    f'vLLM 0.19.x 권장 (현재: {vllm.__version__}). '
    '다른 버전은 AttentionBackend 내부 API 가 다를 수 있음.'
)

props = torch.cuda.get_device_properties(0)
print(f'SMs: {props.multi_processor_count}')
print(f'compute capability: {torch.cuda.get_device_capability(0)}')

## Qwen3-0.6B 구조 요약

| 항목 | 값 |
|---|---|
| hidden_size | 1024 |
| Q heads | 16 |
| KV heads | 8 (GQA 2:1) |
| head_dim | 128 |
| layers | 28 |

Decode 시 grid = B × Hq = 1 × 16 = 16 block. SM 수에 비해 작아 split-KV 의 이득 영역.

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained('Qwen/Qwen3-0.6B')
hd = cfg.head_dim if hasattr(cfg, 'head_dim') else cfg.hidden_size // cfg.num_attention_heads
print('hidden:', cfg.hidden_size)
print('Q heads:', cfg.num_attention_heads, '/ KV heads:', cfg.num_key_value_heads)
print('head_dim:', hd)
print('layers:', cfg.num_hidden_layers)

## Triton 커널 역할 (vllm_split_v2)

`triton_attn.py` 에는 **네 개의** `@triton.jit` 커널:

| 커널 | 호출 시점 | 비고 |
|---|---|---|
| `_fwd_kernel_prefill` | q_len == kv_len, causal | vllm_split 와 동일 |
| `_fwd_kernel_decode_simple` | q_len=1, KV_SPLITS=1 fast path | grid 가 SM 채우면 사용 |
| `_fwd_kernel_decode_partial` | q_len=1, split-KV phase-1 | KV chunk 별 partial m/l/acc |
| `_fwd_kernel_decode_reduce` | split-KV phase-2 | log-sum-exp 안정 결합 |

Wrapper `triton_attention_decode(q, k, v)` 가 `_choose_kv_splits()` 휴리스틱으로 KV_SPLITS 결정 → 1 이면 simple, ≥2 이면 partial+reduce 두 커널 launch.

In [ ]:
from triton_attn import triton_attention_decode, _choose_kv_splits

print(triton_attention_decode.__doc__)
print()
# Heuristic preview for Qwen3 (B=1, Hq=16) at various S
B, Hq = 1, 16
print(f'KV_SPLITS heuristic for B={B}, Hq={Hq} (grid_bh = {B*Hq}):')
for S in (32, 256, 512, 1024, 4096, 16384):
    ns = _choose_kv_splits(B * Hq, S)
    print(f'  S={S:>6}  →  KV_SPLITS={ns}')

In [ ]:
# Smoke test: decode correctness across split counts vs SDPA reference
import torch.nn.functional as F
from triton_attn import triton_attention_prefill, triton_attention_decode

B, Hq, Hkv, D = 1, 16, 8, 128

print('prefill (q_len == kv_len, causal)')
for S in (128, 1024):
    q = torch.randn(B, Hq, S, D, dtype=torch.float16, device='cuda')
    k = torch.randn(B, Hkv, S, D, dtype=torch.float16, device='cuda')
    v = torch.randn(B, Hkv, S, D, dtype=torch.float16, device='cuda')
    ref = F.scaled_dot_product_attention(
        q, k.repeat_interleave(Hq // Hkv, dim=1),
        v.repeat_interleave(Hq // Hkv, dim=1), is_causal=True,
    )
    ours = triton_attention_prefill(q, k, v)
    err = (ours.float() - ref.float()).abs().max().item()
    print(f'  S={S:<6} max_err = {err:.6f}  →  {"PASS" if err < 1e-2 else "FAIL"}')

print()
print('decode (q_len=1) — sweep KV_SPLITS to verify split-KV vs SDPA')
for S in (128, 1024, 4096):
    q_full = torch.randn(B, Hq, S, D, dtype=torch.float16, device='cuda')
    k = torch.randn(B, Hkv, S, D, dtype=torch.float16, device='cuda')
    v = torch.randn(B, Hkv, S, D, dtype=torch.float16, device='cuda')
    ref_full = F.scaled_dot_product_attention(
        q_full, k.repeat_interleave(Hq // Hkv, dim=1),
        v.repeat_interleave(Hq // Hkv, dim=1), is_causal=True,
    )
    ref_last = ref_full[:, :, -1:, :]
    q_last = q_full[:, :, -1:, :].contiguous()
    print(f'  S={S}:')
    for ns in (None, 1, 2, 4, 8):
        if ns is not None and ns > S:
            continue
        ours = triton_attention_decode(q_last, k, v, kv_splits=ns)
        err = (ours.float() - ref_last.float()).abs().max().item()
        tag = 'auto' if ns is None else f'ns={ns}'
        print(f'    {tag:<6} max_err = {err:.6f}  →  {"PASS" if err < 1e-2 else "FAIL"}')

## Plugin entry point 로 자동 등록

`pyproject.toml`:
```toml
[project.entry-points."vllm.general_plugins"]
my_split_v2_backend = "triton_attention_backend:register"
```

`pip install -e .` 이후 vLLM 은 메인 · 엔진 코어 · 워커 모두에서 `register()` 를 자동 호출.

실행 증거는 엔진 코어 stderr 의 `MyTritonImpl[v2].forward fired ...` 로그로 확인 — `[v2]` 태그가 붙어 vllm_split 와 구분됨.

In [ ]:
from importlib.metadata import entry_points

eps = list(entry_points(group='vllm.general_plugins'))
mine = [e for e in eps if e.name == 'my_split_v2_backend']
assert mine, (
    '`my_split_v2_backend` entry point 가 보이지 않는다. '
    '노트북 디렉토리에서 `pip install -e .` 을 실행했는지 확인하라.'
)
print('plugin registered:', mine[0].name, '->', mine[0].value)

In [ ]:
from vllm import LLM, SamplingParams
from vllm.v1.attention.backends.registry import AttentionBackendEnum

llm = LLM(
    model='Qwen/Qwen3-0.6B',
    dtype='float16',
    attention_backend=AttentionBackendEnum.CUSTOM,
    enforce_eager=True,
    max_num_seqs=1,
    max_model_len=2048,
)

In [ ]:
out = llm.generate(
    ['The capital of France is'],
    SamplingParams(temperature=0, max_tokens=32)
)
print(out[0].outputs[0].text)

## 검증: 실제로 split-KV decode 가 실행됐는가

위 `llm.generate(...)` 출력 위에 다음 두 줄이 찍혀야 한다:

```
(EngineCore pid=XXXXX) WARNING ... MyTritonImpl[v2].forward fired (prefill) tokens=5
(EngineCore pid=XXXXX) WARNING ... MyTritonImpl[v2].forward fired (decode, split-KV) s_len=6
```

두 fired 가 모두 찍히면 prefill 과 split-KV decode 양쪽이 실제로 호출된 증거. `[v2]` 태그가 붙어 vllm_split 의 fired 와 구분됨.

주의: `max_tokens=32` 정도의 짧은 generation 에선 decode 의 KV 길이가 작아 wrapper 의 휴리스틱이 KV_SPLITS=1 (simple kernel) 을 고를 수 있다. 그래도 *split-KV 경로의 wrapper* 는 호출됐고 단지 fast path 로 분기한 것.

In [ ]:
from vllm.v1.attention.backends.registry import AttentionBackendEnum

path = AttentionBackendEnum.CUSTOM.get_path()
print('CUSTOM slot ->', path)
assert 'MyTritonBackend' in path, 'CUSTOM slot 에 우리 백엔드가 등록되지 않음'
print('OK — 노트북 프로세스에서도 plugin 자동 등록 확인')
print()
print('실제 split-KV decode 실행 증거는 위 cell 11 출력 바로 위의 `[v2].forward fired (decode, split-KV)` 로그로 확인.')

## 다음 단계

- **paged KV 직접 read 와 결합** — vllm_paged 의 `block_table` indirect 위에 split-KV 를 올린 형태. 현재 v2 는 `_gather_kv` 로 dense 화 후 처리해서 학습용엔 깔끔하지만 production 으론 비효율
- **multi-seq batch 지원** — 현재 `query_start_loc[1] - query_start_loc[0]` 분기는 batch=1 가정. multiseq 변형이 필요
- **GQA KV-head grid 접기** (`BLOCK_M = BLOCK_Q × n_rep`) — vLLM v1 production 에서 추가 ~2× speedup
- **autotune** — `_choose_kv_splits` 의 휴리스틱을 측정 기반 lookup table 로 교체